# Genotype–Phenotype Heterogeneity Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
Dataset is described via a Croissant schema URL. The FAIR^2 package contains structured tabulations of clinical, imaging, and genetic features for three patients.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic info about dataset
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print("Keywords:", metadata.keywords)


## 2. Data Overview
Review available record sets and fields and their IDs.

The FAIR^2 dataset contains multiple record sets. We will enumerate their `@id` and summarize their fields using Croissant metadata.

In [ ]:
# Get record sets
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the metadata (as per the direct metadata). Attempting to infer from the schema...")
    # Try extracting record sets via Croissant API
    # mlcroissant stores record sets in dataset.record_sets
    record_sets = dataset.record_sets

print("Record sets in the dataset:")
for rs in record_sets:
    print(f"  @id: {rs['@id']} | Name: {rs.get('name', '(No name)')} | Description: {rs.get('description', '(No description)')}" )
    if 'fields' in rs:
        print("    Fields:")
        for field in rs['fields']:
            print(f"      - @id: {field['@id']} | Name: {field.get('name', '(No name)')}")
    elif 'columns' in rs:
        print("    Columns:")
        for column in rs['columns']:
            print(f"      - @id: {column['@id']} | Name: {column.get('name', '(No name)')}")
    else:
        print("     No fields/columns found.")

## 3. Data Extraction
Load data from specific record sets (tables) into DataFrames for analysis. Use record set and field `@id` from the overview.

For this dataset, the tables correspond to:
- Table 1: Clinical features
- Table 2: Hormone and imaging results
- Table 3: Genetic variants

In [ ]:
# Gather record set @id values
record_set_ids = []
for rs in record_sets:
    record_set_ids.append(rs['@id'])

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id}: Columns:")
        print(df.columns.tolist())
        print(df.head())
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps, such as filtering records, normalizing numeric fields, and grouping data by attributes.

We'll select a numeric field from Table 2 (hormone levels) and demonstrate filtering and normalization. All references use `@id`.

In [ ]:
# Choose Table 2 record set for hormone-level analysis
# Assume that hormone levels are stored in a column with @id 'cr:HormoneLevel'
# Find Table 2 by matching its name or description
table2_id = None
for rs in record_sets:
    if rs.get('name', '').lower().find('hormone') != -1 or rs.get('description', '').lower().find('hormone') != -1:
        table2_id = rs['@id']
        break
if not table2_id:
    table2_id = record_set_ids[1] if len(record_set_ids)>1 else record_set_ids[0]

df2 = dataframes.get(table2_id)
if df2 is not None and not df2.empty:
    print("Table 2 columns:", df2.columns.tolist())
    # Attempt to find a numeric field (e.g. hormone level)
    numeric_field_id = None
    for col in df2.columns:
        if 'hormone' in col.lower() or 'level' in col.lower():
            numeric_field_id = col
            break
    if not numeric_field_id:
        # Default to first numeric column
        numeric_field_id = df2.select_dtypes('number').columns[0] if not df2.select_dtypes('number').empty else df2.columns[0]

    threshold = 10
    # Filter: hormone level > threshold
    filtered_df = df2[df2[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize hormone levels
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by imaging finding (e.g. adrenal ct result), using @id if available
    group_field = None
    for col in df2.columns:
        if 'ct' in col.lower():
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable DataFrame found for Table 2 EDA.")

## 5. Visualization
Visualize distributions and relationships. Example: histogram of hormone levels and scatterplot between hormone level and imaging findings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of hormone levels
if df2 is not None and numeric_field_id is not None:
    plt.figure(figsize=(6,4))
    sns.histplot(df2[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If imaging finding is available, scatter plot
    if group_field:
        plt.figure(figsize=(6,4))
        sns.scatterplot(data=df2, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} vs {group_field}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to load clinical, laboratory, and genetic data provided in FAIR^2 schema format. We:
- Listed record sets and fields using `@id`
- Loaded tables into pandas DataFrames
- Performed basic EDA: filtering hormone-level, normalization, grouping
- Visualized field distributions

**Takeaways:**
- Data are well-structured per Croissant schema, facilitating research reproducibility.
- Using `@id` referencing and schema-driven loading allows robust data processing.
- The small cohort limits generalizability, but schema-based exploration helps extend methods to richer datasets.